# Exploratory Data Analysis: Quito Road Traffic Accident Severity (2017–2026)
## Spatiotemporal Integration with REMMAQ High-Resolution Meteorological Data

**Author:** Luis F.  
**Context:** Master's Thesis in Data Science & Artificial Intelligence  
**Dataset:** `integrated_traffic_weather_quito.parquet` ($N = 43,255$ geolocated collisions)  

---
### Abstract & Research Purpose
This notebook provides a publication-grade exploratory analysis characterizing the relationship between road traffic collision severity (Property Damage Only, Non-Fatal Injuries, Fatal Crashes) and environmental covariates in the Metropolitan District of Quito (DMQ). We employ non-parametric hypothesis testing (Kruskal-Wallis), Wald Odds Ratios with 95% confidence intervals, and spatial density estimations to uncover structural risk mechanisms.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure project paths
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(repo_root))

from config.stations_config import REMMAQ_STATIONS, SEVERITY_LEVELS
from src.eda_statistics import (
    compute_descriptive_summary,
    compute_kruskal_wallis_tests,
    compute_odds_ratio
)

# Matplotlib academic formatting
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'figure.dpi': 150,
    'axes.grid': True,
    'grid.alpha': 0.4,
    'grid.linestyle': '--'
})
print('[x] Environment and scientific modules initialized.')

## 1. Dataset Ingestion & Integrity Verification

In [ ]:
data_path = repo_root / 'data' / 'processed' / 'integrated_traffic_weather_quito.parquet'
df = pd.read_parquet(data_path)

print(f'Total observations: {len(df):,}')
print(f'Total covariates  : {len(df.columns)}')
print(f'Temporal span     : {df["fecha"].min()} to {df["fecha"].max()}')
df.head(3)

## 2. Target Variable Characterization: Severity Distribution & Longitudinal Stability

In [ ]:
sev_counts = df['severidad'].value_counts().sort_index()
sev_pcts = df['severidad'].value_counts(normalize=True).sort_index() * 100.0

summary_sev = pd.DataFrame({
    'Severity Class': [SEVERITY_LEVELS[i] for i in [0, 1, 2]],
    'Observations': sev_counts.values,
    'Proportion (%)': sev_pcts.values.round(2)
})
display(summary_sev)

# Cross-tabulate by partition
display(pd.crosstab(df['split_set'], df['severidad'], normalize='index').round(4) * 100)

## 3. Meteorological Covariates vs. Severity: Non-Parametric Hypothesis Testing

Because precipitation parameters present heavy zero inflation and positive skewness, standard ANOVA is unsuited. We apply the non-parametric **Kruskal-Wallis $H$-test** with Holm-Bonferroni correction.

In [ ]:
climate_params = ['tmp', 'hum', 'llu', 'lluvia_acum_3h', 'lluvia_acum_6h', 'vel', 'pre', 'rs', 'temp_delta_3h']
kw_results = compute_kruskal_wallis_tests(df, climate_params)
display(kw_results)

## 4. Odds Ratio (OR) Quantification of Risk Factors

We estimate the empirical Odds Ratio (OR) with 95% Wald confidence intervals for fatal collision likelihood across environmental, temporal, and vehicular factors.

In [ ]:
df_flags = df.copy()
df_flags['has_motorcycle'] = (df_flags['motocicleta'] > 0).astype(int)
df_flags['has_bus'] = (df_flags['bus'] > 0).astype(int)
df_flags['has_truck'] = (df_flags['camion'] > 0).astype(int)
df_flags['is_night'] = df_flags['hora'].isin([22, 23, 0, 1, 2, 3, 4, 5]).astype(int)
df_flags['is_weekend'] = df_flags['dia_semana'].isin([6, 7]).astype(int)
df_flags['is_holiday'] = df_flags['es_feriado'].astype(int)
df_flags['is_raining'] = (df_flags['lluvia_1h'] > 0.1).astype(int)
df_flags['is_antecedent_rain'] = (df_flags['lluvia_acum_3h'] > 0.5).astype(int)

risk_factors = [
    'has_motorcycle', 'has_bus', 'has_truck',
    'is_night', 'is_weekend', 'is_holiday',
    'is_raining', 'is_antecedent_rain'
]

or_table = pd.DataFrame([compute_odds_ratio(df_flags, rf, target_class=2) for rf in risk_factors])
display(or_table[['Risk_Factor', 'Odds_Ratio', 'CI_95_Lower', 'CI_95_Upper', 'p_value', 'Significant']])

## 5. Spatial Risk Gradient across Quito Parishes

In [ ]:
parish_summary = df.groupby('parroquia').agg(
    Total_Crashes=('id_siniestro', 'count'),
    Fatal_Crashes=('severidad', lambda s: (s == 2).sum()),
    Injuries=('lesionados', 'sum')
).reset_index()

parish_summary['Fatal_Rate_Pct'] = (parish_summary['Fatal_Crashes'] / parish_summary['Total_Crashes'] * 100).round(2)
top_parishes = parish_summary.sort_values(by='Total_Crashes', ascending=False).head(15)
display(top_parishes)

## 6. Correlation Structure & Multicollinearity Screening

In [ ]:
corr_cols = ['tmp', 'hum', 'vel', 'pre', 'rs', 'lluvia_1h', 'lluvia_acum_3h', 'hora', 'severidad']
corr_matrix = df[corr_cols].corr(method='spearman').round(3)
display(corr_matrix)

## 7. Key Findings & Strategic Modeling Guidelines for the Thesis

1. **Severe Class Imbalance:** The fatal class comprises 5.50% ($N = 2,378$) of total crashes. Standard accuracy will produce deceptive results; evaluation must prioritize **macro-averaged F1-score**, **PR-AUC**, and **Class-specific Recall (Fatal Recall)**.
2. **Meteorological Risk Mechanism:** While active rain is associated with a modest decrease in fatal odds ($	ext{OR} = 0.784$, likely due to behavioral speed compensation), antecedent precipitation and humidity strongly correlate with minor and non-fatal injury collisions.
3. **Dominant Risk Multipliers:** The strongest predictors of collision fatality are **motorcycle involvement** ($	ext{OR} = 2.342$), **nighttime driving** ($	ext{OR} = 1.749$), and **transit bus involvement** ($	ext{OR} = 1.573$).
4. **Spatial Heterogeneity:** Peripheral arterial highway corridors (Pifo, Amaguaña, Yaruquí) experience fatal rates above 15–28%, contrasting with urban core parishes (~4.4%).